# Model Debug

Forward pass sanity checks.

In [ ]:
{
 "cells": [
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": ["# SignVoice — Model Debug\nTest architecture, forward pass, and training diagnostics."]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "import sys\n",
    "sys.path.insert(0, '..')\n",
    "\n",
    "import torch\n",
    "import torch.nn as nn\n",
    "import yaml\n",
    "import numpy as np\n",
    "import matplotlib.pyplot as plt\n",
    "from pathlib import Path\n",
    "\n",
    "device = 'cuda' if torch.cuda.is_available() else 'cpu'\n",
    "print(f'Device  : {device}')\n",
    "if device == 'cuda':\n",
    "    print(f'GPU     : {torch.cuda.get_device_name(0)}')\n",
    "    print(f'Memory  : {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB')\n",
    "print(f'PyTorch : {torch.__version__}')"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": ["## 1. Classifier Architecture"]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "class SignClassifier(nn.Module):\n",
    "    def __init__(self, input_dim=183, d_model=64,\n",
    "                 n_heads=2, n_layers=1, n_classes=10):\n",
    "        super().__init__()\n",
    "        self.proj = nn.Linear(input_dim, d_model)\n",
    "        self.bn   = nn.BatchNorm1d(d_model)\n",
    "        layer     = nn.TransformerEncoderLayer(\n",
    "            d_model=d_model, nhead=n_heads,\n",
    "            dim_feedforward=128, dropout=0.5,\n",
    "            batch_first=True, norm_first=True,\n",
    "        )\n",
    "        self.transformer = nn.TransformerEncoder(layer, num_layers=n_layers)\n",
    "        self.classifier  = nn.Sequential(\n",
    "            nn.Dropout(0.5),\n",
    "            nn.Linear(d_model, 32),\n",
    "            nn.ReLU(),\n",
    "            nn.Dropout(0.3),\n",
    "            nn.Linear(32, n_classes),\n",
    "        )\n",
    "\n",
    "    def forward(self, x, mask=None):\n",
    "        B, T, F = x.shape\n",
    "        x = self.proj(x)\n",
    "        x = self.bn(x.reshape(B*T, -1)).reshape(B, T, -1)\n",
    "        x = self.transformer(x, src_key_padding_mask=mask)\n",
    "        x = x.mean(dim=1)\n",
    "        return self.classifier(x)\n",
    "\n",
    "model  = SignClassifier(n_classes=10).to(device)\n",
    "total  = sum(p.numel() for p in model.parameters())\n",
    "train_ = sum(p.numel() for p in model.parameters() if p.requires_grad)\n",
    "\n",
    "print('SignClassifier Architecture')\n",
    "print('=' * 40)\n",
    "for name, module in model.named_children():\n",
    "    params = sum(p.numel() for p in module.parameters())\n",
    "    print(f'  {name:15s} : {params:>8,} params')\n",
    "print('=' * 40)\n",
    "print(f'  Total      : {total:>8,}')\n",
    "print(f'  Trainable  : {train_:>8,}')\n",
    "print(f'  Size       : {total * 4 / 1024**2:.2f} MB')"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": ["## 2. Forward Pass Test"]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "B, T, F = 4, 45, 183\n",
    "x       = torch.randn(B, T, F).to(device)\n",
    "\n",
    "model.eval()\n",
    "with torch.no_grad():\n",
    "    out = model(x)\n",
    "\n",
    "print(f'Input  : {x.shape}')\n",
    "print(f'Output : {out.shape}  (B, n_classes)')\n",
    "print(f'Probs  : {torch.softmax(out[0], dim=0).cpu().numpy().round(3)}')\n",
    "print('Forward pass: OK')"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": ["## 3. Load Trained Classifier"]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "ckpt_path = Path('../checkpoints/classifier.pt')\n",
    "\n",
    "if not ckpt_path.exists():\n",
    "    print('No classifier found. Run: python scripts/test_inference.py')\n",
    "else:\n",
    "    ckpt    = torch.load(ckpt_path, map_location=device, weights_only=False)\n",
    "    glosses = ckpt['glosses']\n",
    "    print(f'Glosses    : {glosses}')\n",
    "    print(f'N classes  : {len(glosses)}')\n",
    "\n",
    "    model = SignClassifier(n_classes=len(glosses)).to(device)\n",
    "    model.load_state_dict(ckpt['model'])\n",
    "    model.eval()\n",
    "    print('Classifier loaded successfully')"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": ["## 4. Training Loss Curves"]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "from tensorboard.backend.event_processing.event_accumulator import EventAccumulator\n",
    "\n",
    "runs_dir = Path('../runs')\n",
    "run_dirs = sorted(runs_dir.glob('*')) if runs_dir.exists() else []\n",
    "\n",
    "if not run_dirs:\n",
    "    print('No training runs found.')\n",
    "else:\n",
    "    latest = run_dirs[-1]\n",
    "    print(f'Run: {latest.name}')\n",
    "\n",
    "    ea   = EventAccumulator(str(latest))\n",
    "    ea.Reload()\n",
    "    tags = ea.Tags()['scalars']\n",
    "    print(f'Tags: {tags}')\n",
    "\n",
    "    fig, axes = plt.subplots(1, 2, figsize=(14, 4))\n",
    "\n",
    "    if 'train/loss' in tags:\n",
    "        events = ea.Scalars('train/loss')\n",
    "        steps  = [e.step  for e in events]\n",
    "        loss   = [e.value for e in events]\n",
    "        axes[0].plot(steps, loss, color='steelblue', linewidth=1)\n",
    "        axes[0].set_title(f'Train loss (final={loss[-1]:.4f})')\n",
    "        axes[0].set_xlabel('Step')\n",
    "        axes[0].set_ylabel('Loss')\n",
    "        axes[0].grid(True, alpha=0.3)\n",
    "\n",
    "    if 'val/loss' in tags:\n",
    "        events = ea.Scalars('val/loss')\n",
    "        epochs = [e.step  for e in events]\n",
    "        loss   = [e.value for e in events]\n",
    "        axes[1].plot(epochs, loss, color='coral',\n",
    "                     linewidth=1.5, marker='o', markersize=4)\n",
    "        axes[1].set_title(f'Val loss (final={loss[-1]:.4f})')\n",
    "        axes[1].set_xlabel('Epoch')\n",
    "        axes[1].set_ylabel('Loss')\n",
    "        axes[1].grid(True, alpha=0.3)\n",
    "\n",
    "    plt.tight_layout()\n",
    "    plt.show()"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": ["## 5. Confusion Matrix on Test Set"]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "import json\n",
    "import sys\n",
    "sys.path.insert(0, '..')\n",
    "from src.preprocessing.normalizer import KeypointNormalizer\n",
    "\n",
    "ckpt_path = Path('../checkpoints/classifier.pt')\n",
    "if not ckpt_path.exists():\n",
    "    print('No classifier found.')\n",
    "else:\n",
    "    ckpt    = torch.load(ckpt_path, map_location=device, weights_only=False)\n",
    "    glosses = ckpt['glosses']\n",
    "    model   = SignClassifier(n_classes=len(glosses)).to(device)\n",
    "    model.load_state_dict(ckpt['model'])\n",
    "    model.eval()\n",
    "\n",
    "    normalizer = KeypointNormalizer('../data/processed/keypoint_stats.npz')\n",
    "    normalizer.load()\n",
    "\n",
    "    with open('../data/processed/test_manifest.json') as f:\n",
    "        test_data = json.load(f)\n",
    "\n",
    "    g_to_idx = {g: i for i, g in enumerate(glosses)}\n",
    "    y_true, y_pred = [], []\n",
    "\n",
    "    for s in test_data:\n",
    "        if s['gloss'] not in g_to_idx:\n",
    "            continue\n",
    "        kp   = normalizer.normalize(np.load(s['keypoint_file']))\n",
    "        kp_t = torch.from_numpy(kp).unsqueeze(0).to(device)\n",
    "        with torch.no_grad():\n",
    "            pred = model(kp_t).argmax(dim=1).item()\n",
    "        y_true.append(g_to_idx[s['gloss']])\n",
    "        y_pred.append(pred)\n",
    "\n",
    "    from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay\n",
    "    cm  = confusion_matrix(y_true, y_pred)\n",
    "    acc = sum(t == p for t, p in zip(y_true, y_pred)) / len(y_true) * 100\n",
    "\n",
    "    fig, ax = plt.subplots(figsize=(10, 8))\n",
    "    disp = ConfusionMatrixDisplay(\n",
    "        confusion_matrix=cm, display_labels=glosses\n",
    "    )\n",
    "    disp.plot(ax=ax, cmap='Blues', colorbar=False)\n",
    "    ax.set_title(f'Confusion Matrix — Accuracy: {acc:.1f}%')\n",
    "    plt.xticks(rotation=45, ha='right')\n",
    "    plt.tight_layout()\n",
    "    plt.show()\n",
    "    print(f'Overall accuracy: {acc:.1f}%')"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": ["## 6. Per-Sign Accuracy Bar Chart"]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "if 'y_true' in dir() and len(y_true) > 0:\n",
    "    from collections import defaultdict\n",
    "    gloss_correct = defaultdict(list)\n",
    "\n",
    "    for t, p in zip(y_true, y_pred):\n",
    "        gloss_correct[glosses[t]].append(t == p)\n",
    "\n",
    "    signs     = sorted(gloss_correct.keys())\n",
    "    accs      = [sum(gloss_correct[g]) / len(gloss_correct[g]) * 100\n",
    "                 for g in signs]\n",
    "    colors    = ['green' if a >= 60 else 'orange' if a >= 40\n",
    "                 else 'red' for a in accs]\n",
    "\n",
    "    fig, ax = plt.subplots(figsize=(12, 5))\n",
    "    bars = ax.bar(signs, accs, color=colors, edgecolor='white')\n",
    "    ax.axhline(60, color='green', linestyle='--',\n",
    "               linewidth=1, label='60% threshold')\n",
    "    ax.set_xlabel('Sign')\n",
    "    ax.set_ylabel('Accuracy (%)')\n",
    "    ax.set_title('Per-sign accuracy on test set')\n",
    "    ax.set_ylim(0, 105)\n",
    "    ax.legend()\n",
    "\n",
    "    for bar, acc in zip(bars, accs):\n",
    "        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,\n",
    "                f'{acc:.0f}%', ha='center', va='bottom', fontsize=9)\n",
    "\n",
    "    plt.xticks(rotation=45, ha='right')\n",
    "    plt.tight_layout()\n",
    "    plt.show()\n",
    "\n",
    "    print('Best signs for demo (above 60%):',\n",
    "          [s for s, a in zip(signs, accs) if a >= 60])"
   ]
  }
 ],
 "metadata": {
  "kernelspec": {
   "display_name": "SignVoice (venv311)",
   "language": "python",
   "name": "signvoice"
  },
  "language_info": {
   "name": "python",
   "version": "3.11.9"
  }
 },
 "nbformat": 4,
 "nbformat_minor": 4
}